# INF 791 - Tópicos Especiais II – Redes Complexas
## Trabalho Prático - Intermediário

Este notebook foi desenvolvido para resolver passo a passo as questões do trabalho prático.


# 1. Coleta de Dados

## (a) Escolha do Sistema e Construção do Coletor do Grafo

### Base de dados escolhida  
A base de dados escolhida para esta análise é a Social circles: Facebook (*Facebook Ego Networks*), obtida através da coleção pública *SNAP (Stanford Large Network Dataset Collection)*.  

Uma ego-network é a rede que se forma quando pegamos um usuário (o "Ego"), todos os seus amigos diretos, e todas as amizades que existem entre esses amigos. Isso significa que o grafo não é um recorte aleatório, mas sim um mapa detalhado de como círculos sociais inteiros e grupos de amigos operam em conjunto.
 
A proposta em relação a este grafo é comprovar empiricamente propriedades clássicas e leis universais das redes sociais reais estudadas em sala de aula. Partindo da rede formada por esse dataset, serão avaliadas as seguintes métricas de redes complexas:

### Grau Médio e Distribuição de Grau  
Esta métrica quantifica a popularidade individual de cada nó, determinando exatamente a quantidade de amigos de um usuário. Será utilizada para expor a desigualdade de popularidade na rede. 

O objetivo com essa métrica é verificar se o grafo apresenta um comportamento Livre de Escala (*Scale-Free*), caracterizado por uma minoria de usuários com um número gigantesco de ligações.

### Coeficiente de Clusterização  
Através do coeficiente de clusterização vamos medir matematicamente a probabilidade de que dois amigos de um determinado usuário também sejam amigos entre si. 

Utilizaremos esse cálculo na análise para comprovar o princípio do "Fechamento Triádico", atestando se a rede do Facebook possui uma tendência à formação de comunidades isoladas, panelinhas e bolhas sociais densas.

### Distância Média  
A distância média será calculada para encontrar o número mínimo de conexões necessárias para que uma informação vá de um usuário qualquer até outro dentro do grafo. 

Essa métrica servirá para comprovar de forma empírica a teoria dos "Seis Graus de Separação" e demonstrar o fenômeno do Mundo Pequeno (*Small-World*).

### Sobreposição de Vizinhança (Overlap)  
O *overlap* medirá a quantidade percentual de amigos que duas pessoas conectadas possuem em comum. Nas análises, essa métrica será o pilar para diferenciar e validar a teoria sociológica dos laços. Conseguiremos identificar quais arestas são "Laços Fortes" e quais são "Laços Fracos".

### Componentes Conexos  
Por fim, o cálculo de componentes conexos identificará se existem ilhas de usuários completamente isoladas do resto do grafo e qual o tamanho exato de cada um desses blocos. A análise partirá dessa contagem para confirmar a tendência das redes de conectarem a vasta maioria de seus participantes em um único "Componente Gigante" contínuo.

### Obtenção dos Dados (API vs Base Pública):  
Historicamente, dados de listas de amigos do Facebook eram extraídos via *Facebook Graph API*. No entanto, devido às grandes restrições de privacidade das plataformas atualmente, essa coleta tornou-se impossível.

Dessa forma o presente trabalho irá explorar a base pública do projeto acadêmico de Stanford (SNAP), que fornece esses dados de conectividade de maneira totalmente anonimizada e em larga escala.
* Fonte dos dados: http://snap.stanford.edu/data/ego-Facebook.html

### Construção do Coletor e Chamadas Utilizadas:  
Para automatizar a coleta a partir do repositório público, construímos um script que realiza as seguintes operações:
1. `requests.get()`: Para fazer a requisição HTTP e baixar os dados brutos compactados.
2. `gzip` e `shutil`: Para manipular e descompactar o arquivo `.gz`.
3. `nx.read_edgelist()`: Para ler o arquivo de texto extraído e estruturar as amizades como um objeto grafo não-direcionado na biblioteca *NetworkX*.


In [ ]:
import networkx as nx
import requests
import gzip
import shutil
import os

# Configurações de URL e arquivos locais
url = "https://snap.stanford.edu/data/facebook_combined.txt.gz"
gz_file = "facebook_combined.txt.gz"
txt_file = "facebook_combined.txt"

# 1. Coleta / Download dos dados brutos via HTTP
if not os.path.exists(txt_file):
    print("Iniciando coleta a partir do repositório público (SNAP)...")
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(gz_file, 'wb') as f_out:
            shutil.copyfileobj(response.raw, f_out)
        print("Download dos dados concluído!")
        
        # 2. Descompactação
        print("Extraindo o dataset...")
        with gzip.open(gz_file, 'rb') as f_in:
            with open(txt_file, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        print("Extração concluída!")
    else:
        print(f"Falha ao baixar o arquivo. Status code: {response.status_code}")
else:
    print("O dataset já foi coletado e está presente no diretório local.")

# 3. Processamento: Construção do Grafo Não-Direcionado
print("\nMontando a rede complexa na memória...")
G = nx.read_edgelist(txt_file, create_using=nx.Graph(), nodetype=int)

print(f"Grafo construído com sucesso!")
print(f"-> Total de Nós (Usuários anonimizados): {G.number_of_nodes()}")
print(f"-> Total de Arestas (Relações de amizade): {G.number_of_edges()}")
